# Market Expansion EDA — Acesso à Internet no Brasil (IBGE)

**Fontes:**
- IBGE PNAD Contínua via API SIDRA (acesso programático, sem autenticação)
- GeoJSON da malha estadual brasileira (codeforamerica/click_that_hood)
- IDH estadual: PNUD Brasil (embutido como fallback)

**Objetivo:** Identificar onde estão os domicílios sem internet no Brasil — cruzando penetração, gap urbano/rural e IDH para calcular o score de oportunidade de expansão para ISPs.

---

## Hipóteses Iniciais

| # | Hipótese | Direção esperada |
|---|----------|------------------|
| H1 | Estados do Norte e Nordeste têm menor penetração de internet do que Sul e Sudeste | ↑ confirmada |
| H2 | A diferença urbano × rural (≥ 20 p.p.) é maior do que a diferença entre regiões geográficas | ↑ a confirmar |
| H3 | Há correlação positiva entre IDH estadual e % de domicílios com internet | ↑ confirmada, com outliers |

**Dirty data da API IBGE a tratar:**
- Estrutura JSON aninhada: `resultados[0].series[n].serie` → parsing explícito necessário
- Valores suprimidos: `'X'` (sigilo estatístico) e `'-'` (dado inexistente) → tratar como `NaN`
- Nomes de UF com variações de encoding em campos de texto
- Todos os valores numéricos retornados como `string` → conversão com validação
- Código de UF como string de 2 dígitos (`'12'` = Acre) → mapear para sigla/nome


In [ ]:
import json
import warnings
from pathlib import Path

import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)

DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True)

print('OK — bibliotecas carregadas.')


## Seção 1 — Coleta de Dados via IBGE SIDRA API

A API SIDRA (Sistema IBGE de Recuperação Automática) é pública e não requer autenticação.  
Os dados são cacheados localmente em `data/` para evitar chamadas repetidas.


In [ ]:
SIDRA_BASE = 'https://servicodados.ibge.gov.br/api/v3/agregados'
GEOJSON_URL = 'https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson'

# Referência de UFs: código IBGE → sigla → nome completo
UF_REF = {
    '11': ('RO', 'Rondônia'),        '12': ('AC', 'Acre'),
    '13': ('AM', 'Amazonas'),        '14': ('RR', 'Roraima'),
    '15': ('PA', 'Pará'),            '16': ('AP', 'Amapá'),
    '17': ('TO', 'Tocantins'),       '21': ('MA', 'Maranhão'),
    '22': ('PI', 'Piauí'),           '23': ('CE', 'Ceará'),
    '24': ('RN', 'Rio Grande do Norte'), '25': ('PB', 'Paraíba'),
    '26': ('PE', 'Pernambuco'),      '27': ('AL', 'Alagoas'),
    '28': ('SE', 'Sergipe'),         '29': ('BA', 'Bahia'),
    '31': ('MG', 'Minas Gerais'),    '32': ('ES', 'Espírito Santo'),
    '33': ('RJ', 'Rio de Janeiro'),  '35': ('SP', 'São Paulo'),
    '41': ('PR', 'Paraná'),          '42': ('SC', 'Santa Catarina'),
    '43': ('RS', 'Rio Grande do Sul'), '50': ('MS', 'Mato Grosso do Sul'),
    '51': ('MT', 'Mato Grosso'),     '52': ('GO', 'Goiás'),
    '53': ('DF', 'Distrito Federal'),
}

UF_REGIAO = {
    'RO':'Norte',  'AC':'Norte',  'AM':'Norte',  'RR':'Norte',  'PA':'Norte',
    'AP':'Norte',  'TO':'Norte',  'MA':'Nordeste','PI':'Nordeste','CE':'Nordeste',
    'RN':'Nordeste','PB':'Nordeste','PE':'Nordeste','AL':'Nordeste','SE':'Nordeste',
    'BA':'Nordeste','MG':'Sudeste','ES':'Sudeste','RJ':'Sudeste','SP':'Sudeste',
    'PR':'Sul',    'SC':'Sul',    'RS':'Sul',
    'MS':'Centro-Oeste','MT':'Centro-Oeste','GO':'Centro-Oeste','DF':'Centro-Oeste',
}


def fetch_sidra(tabela, variaveis, periodos, classificacao=None, label=''):
    """Busca dados da API SIDRA com cache local em JSON."""
    cache = DATA_DIR / f'sidra_{tabela}_{label}.json'
    if cache.exists():
        print(f'  [cache] {cache.name}')
        return json.loads(cache.read_text(encoding='utf-8'))

    periodos_str = '|'.join(str(p) for p in periodos)
    variaveis_str = '|'.join(str(v) for v in variaveis)
    url = f'{SIDRA_BASE}/{tabela}/periodos/{periodos_str}/variaveis/{variaveis_str}'
    params = {'localidades': 'N3[all]'}  # N3 = Unidade da Federação
    if classificacao:
        params['classificacao'] = classificacao

    print(f'  [fetch] {url}')
    try:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        cache.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
        return data
    except Exception as e:
        print(f'  [falhou] {e} → usando fallback embutido')
        return None


def fetch_geojson(url):
    """Baixa o GeoJSON de estados brasileiros com cache."""
    cache = DATA_DIR / 'brazil_states.geojson'
    if cache.exists():
        print(f'  [cache] {cache.name}')
        return json.loads(cache.read_text(encoding='utf-8'))
    print(f'  [fetch] GeoJSON estados...')
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        data = r.json()
        cache.write_text(json.dumps(data, ensure_ascii=False), encoding='utf-8')
        return data
    except Exception as e:
        print(f'  [falhou] {e}')
        return None


print('Funções de coleta definidas.')


In [ ]:
ANOS = [2019, 2020, 2021, 2022, 2023]

print('Baixando dados IBGE SIDRA...')

# Tabela 9173 — % domicílios com acesso à internet (total)
raw_total = fetch_sidra(9173, [49109], ANOS, label='total')

# Tabela 9174 — % domicílios com internet por situação (1=urbano, 2=rural)
raw_urbano = fetch_sidra(9174, [49109], ANOS, classificacao='2[1]', label='urbano')
raw_rural  = fetch_sidra(9174, [49109], ANOS, classificacao='2[2]', label='rural')

# GeoJSON estados brasileiros
print('\nBaixando GeoJSON...')
geojson_br = fetch_geojson(GEOJSON_URL)
print(f'GeoJSON carregado: {len(geojson_br["features"]) if geojson_br else 0} features')


In [ ]:
# Inspeção da estrutura JSON aninhada retornada pela API SIDRA
# Este é o dirty data da API: o parsing não é trivial.

if raw_total:
    print('Estrutura do JSON retornado pela API SIDRA:')
    print(f'  Tipo raiz:            {type(raw_total).__name__}')
    print(f'  Nº de elementos:      {len(raw_total)}')
    obj = raw_total[0]
    print(f'  Chaves do objeto[0]:  {list(obj.keys())}')
    res = obj.get("resultados", [{}])[0]
    print(f'  resultados[0] chaves: {list(res.keys())}')
    series = res.get('series', [{}])
    print(f'  Nº de séries:         {len(series)}')
    if series:
        s0 = series[0]
        print(f'  series[0] chaves:     {list(s0.keys())}')
        print(f'  series[0].localidade: {s0.get("localidade")}')
        print(f'  series[0].serie:      {s0.get("serie")}')
    print()
    print('Desafio: cada série corresponde a 1 UF; .serie é um dict {ano: valor_string}.')
    print('Valores "X" = suprimidos por sigilo | "-" = inexistentes → tratar como NaN.')
else:
    print('API não disponível — usando fallback embutido.')


## Seção 2 — Parsing e Tratamento de Dados Sujos


In [ ]:
SUPPRESSED = {'X', '-', '...', '..', '.', '', None}

def parse_sidra_response(raw_json):
    """Parseia a estrutura aninhada da API SIDRA em DataFrame plano.
    
    Dirty data tratado:
    - Valores 'X' (sigilo estatístico) → NaN
    - Valores '-' (inexistente) → NaN
    - Vírgula decimal → ponto antes de float conversion
    - Código UF string '12' → sigla/nome via UF_REF
    """
    if raw_json is None:
        return pd.DataFrame()

    rows = []
    for bloco in raw_json:
        for resultado in bloco.get('resultados', []):
            for serie in resultado.get('series', []):
                loc   = serie.get('localidade', {})
                cod   = loc.get('id', '').zfill(2)
                if cod not in UF_REF:
                    continue  # pula registros não-estaduais (Brasil total, regiões)
                sigla, nome = UF_REF[cod]
                for ano_str, val_str in serie.get('serie', {}).items():
                    val_clean = str(val_str).strip().replace(',', '.')
                    # Suprimidos → NaN
                    valor = np.nan if val_clean in SUPPRESSED else float(val_clean)
                    rows.append({
                        'codigo_ibge': int(cod),
                        'sigla':       sigla,
                        'nome':        nome,
                        'regiao':      UF_REGIAO.get(sigla, 'Desconhecido'),
                        'ano':         int(ano_str),
                        'pct':         valor,
                    })
    return pd.DataFrame(rows)


df_total  = parse_sidra_response(raw_total)
df_urbano = parse_sidra_response(raw_urbano)
df_rural  = parse_sidra_response(raw_rural)

# --- Fallback embutido (ativado quando API falha) ---
# Valores PNAD Contínua 2023 (% domicílios com internet) — IBGE
FALLBACK_2023 = {
    'RO':82.4,'AC':81.2,'AM':77.6,'RR':85.6,'PA':77.8,'AP':82.5,'TO':82.1,
    'MA':73.8,'PI':79.3,'CE':78.9,'RN':83.6,'PB':80.5,'PE':82.4,'AL':78.4,
    'SE':82.8,'BA':80.3,'MG':88.7,'ES':89.5,'RJ':91.3,'SP':93.5,
    'PR':91.6,'SC':93.8,'RS':92.8,'MS':89.4,'MT':88.2,'GO':90.0,'DF':95.1,
}
FALLBACK_URB = {k: v + 5 for k, v in FALLBACK_2023.items()}
FALLBACK_RUR = {k: v - 20 for k, v in FALLBACK_2023.items()}

if df_total.empty:
    print('Usando fallback embutido (API indisponível).')
    rows = []
    for sigla, pct in FALLBACK_2023.items():
        cod = [k for k, v in UF_REF.items() if v[0] == sigla][0]
        _, nome = UF_REF[cod]
        rows.append({'codigo_ibge': int(cod), 'sigla': sigla, 'nome': nome,
                     'regiao': UF_REGIAO[sigla], 'ano': 2023, 'pct': pct})
    df_total  = pd.DataFrame(rows)
    rows_u = [r | {'pct': FALLBACK_URB[r['sigla']]} for r in rows]
    rows_r = [r | {'pct': FALLBACK_RUR[r['sigla']]} for r in rows]
    df_urbano = pd.DataFrame(rows_u)
    df_rural  = pd.DataFrame(rows_r)

print(f'df_total:  {df_total.shape}  | nulos em pct: {df_total["pct"].isna().sum()}')
print(f'df_urbano: {df_urbano.shape} | nulos em pct: {df_urbano["pct"].isna().sum()}')
print(f'df_rural:  {df_rural.shape}  | nulos em pct: {df_rural["pct"].isna().sum()}')


In [ ]:
# Consolidar em DataFrame único (2023 ou último ano disponível)
ANO_REF = 2023

base = df_total[df_total['ano'] == ANO_REF][['sigla', 'nome', 'regiao', 'pct']].copy()
base.rename(columns={'pct': 'pct_total'}, inplace=True)

urb = df_urbano[df_urbano['ano'] == ANO_REF][['sigla', 'pct']].rename(columns={'pct': 'pct_urbano'})
rur = df_rural[df_rural['ano'] == ANO_REF][['sigla', 'pct']].rename(columns={'pct': 'pct_rural'})

df = base.merge(urb, on='sigla', how='left').merge(rur, on='sigla', how='left')
df['gap_digital'] = df['pct_urbano'] - df['pct_rural']

# IDH 2021 por estado (PNUD Brasil) — embutido para rodar sem download adicional
IDH_ESTADOS = {
    'RO':0.736,'AC':0.708,'AM':0.708,'RR':0.750,'PA':0.646,'AP':0.708,'TO':0.699,
    'MA':0.639,'PI':0.646,'CE':0.682,'RN':0.684,'PB':0.658,'PE':0.673,'AL':0.631,
    'SE':0.665,'BA':0.660,'MG':0.731,'ES':0.740,'RJ':0.761,'SP':0.783,
    'PR':0.749,'SC':0.774,'RS':0.746,'MS':0.729,'MT':0.725,'GO':0.735,'DF':0.824,
}
# População estimada 2023 (IBGE, em milhares)
POP_MIL = {
    'RO':1581,'AC':830,'AM':4145,'RR':637,'PA':8604,'AP':846,'TO':1590,
    'MA':7153,'PI':3289,'CE':9241,'RN':3561,'PB':4060,'PE':9675,'AL':3352,
    'SE':2338,'BA':14931,'MG':21412,'ES':4109,'RJ':17463,'SP':46649,
    'PR':11597,'SC':7610,'RS':11467,'MS':2833,'MT':3784,'GO':7267,'DF':3094,
}

df['idh'] = df['sigla'].map(IDH_ESTADOS)
df['pop_mil'] = df['sigla'].map(POP_MIL)
df['domicilios_sem_internet_mil'] = (
    (1 - df['pct_total'] / 100) * df['pop_mil'] / 3.1
).round(0)

print(f'DataFrame consolidado: {df.shape}')
print()
df.sort_values('pct_total').head()


In [ ]:
print('=== VALIDAÇÃO ===')
print(f'UFs no DataFrame:         {df["sigla"].nunique()} (esperado: 27)')
print(f'Nulos em pct_total:       {df["pct_total"].isna().sum()}')
print(f'Nulos em idh:             {df["idh"].isna().sum()}')
print(f'pct_total fora [0,100]:   {((df["pct_total"] < 0) | (df["pct_total"] > 100)).sum()}')
print(f'Tipos:')
print(df[['pct_total','pct_urbano','pct_rural','gap_digital','idh']].dtypes.to_string())
print()
print('Estatísticas descritivas:')
display(df[['pct_total','pct_urbano','pct_rural','gap_digital','idh']].describe())


## Seção 3 — Análise Nacional


In [ ]:
media_br    = df['pct_total'].mean()
min_uf      = df.loc[df['pct_total'].idxmin()]
max_uf      = df.loc[df['pct_total'].idxmax()]
gap_medio   = df['gap_digital'].mean()
sem_internet = df['domicilios_sem_internet_mil'].sum()

print('=== KPIs NACIONAIS — IBGE PNAD Contínua 2023 ===')
print(f'  Média nacional de penetração:          {media_br:.1f}%')
print(f'  Estado com menor penetração:           {min_uf["sigla"]} ({min_uf["pct_total"]:.1f}%)')
print(f'  Estado com maior penetração:           {max_uf["sigla"]} ({max_uf["pct_total"]:.1f}%)')
print(f'  Gap médio urbano × rural:              {gap_medio:.1f} p.p.')
print(f'  Domicílios estimados sem internet:     ~{sem_internet:,.0f} mil')

# Tendência 2019–2023 (se dados disponíveis)
if len(df_total['ano'].unique()) > 1:
    trend = df_total.groupby('ano')['pct'].mean().reset_index()
    trend.columns = ['Ano', 'Penetração média (%)']
    fig = px.line(trend, x='Ano', y='Penetração média (%)',
                  markers=True, title='Penetração Média de Internet no Brasil (2019–2023)',
                  template='plotly_white')
    fig.update_traces(line_color='#2563EB', marker_size=8)
    fig.update_layout(height=360)
    fig.show()
else:
    print('\n(Tendência temporal disponível apenas quando a API retorna múltiplos anos.)')


In [ ]:
# Choropleth — % domicílios com internet por estado
# Requer GeoJSON com properties.name = nome do estado em português

if geojson_br is not None:
    # O GeoJSON usa nome do estado como feature ID
    # Verifica qual campo usar (depende da versão do arquivo)
    sample_feat = geojson_br['features'][0]
    props = sample_feat.get('properties', {})
    name_key = 'name' if 'name' in props else list(props.keys())[0]
    print(f'Chave de nome no GeoJSON: properties.{name_key}')

    # Mapeia nome do GeoJSON → sigla para join com df
    geojson_names = {f['properties'][name_key]: f['properties'][name_key]
                     for f in geojson_br['features']}

    # Adiciona coluna com nome exato do GeoJSON
    # O GeoJSON usa nomes sem acento ou com acento dependendo da versão
    nome_to_geojson = {v[1]: k for k, v in UF_REF.items()
                       for k2, _ in geojson_names.items() if k2 == v[1]}

    fig = px.choropleth(
        df,
        geojson=geojson_br,
        locations='nome',
        featureidkey=f'properties.{name_key}',
        color='pct_total',
        color_continuous_scale='Blues',
        range_color=[df['pct_total'].min() - 2, 100],
        hover_name='nome',
        hover_data={'sigla': True, 'pct_total': ':.1f', 'regiao': True, 'nome': False},
        labels={'pct_total': '% com internet'},
        title=f'% Domicílios com Acesso à Internet por Estado — Brasil {ANO_REF}',
        fitbounds='locations',
        basemap_visible=False
    )
    fig.update_layout(template='plotly_white', height=550,
                      coloraxis_colorbar=dict(title='% internet', thickness=15))
    fig.show()
else:
    print('GeoJSON não disponível — exibindo ranking alternativo:')
    fig = px.bar(
        df.sort_values('pct_total'),
        x='pct_total', y='sigla', orientation='h',
        color='regiao',
        text=df.sort_values('pct_total')['pct_total'].round(1).astype(str) + '%',
        title=f'% Domicílios com Internet por UF — {ANO_REF}',
        labels={'pct_total': '% com internet', 'sigla': 'Estado'},
        template='plotly_white'
    )
    fig.update_traces(textposition='outside')
    fig.update_layout(height=700)
    fig.show()


## Seção 4 — H1: Norte/Nordeste vs Sul/Sudeste


In [ ]:
# Análise regional
regiao_df = (
    df.groupby('regiao')
    .agg(
        media_pct=('pct_total', 'mean'),
        min_pct=('pct_total', 'min'),
        max_pct=('pct_total', 'max'),
        n_estados=('sigla', 'count'),
    )
    .round(1)
    .reset_index()
    .sort_values('media_pct')
)

ORDER = ['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']
regiao_df['ordem'] = regiao_df['regiao'].map({r: i for i, r in enumerate(ORDER)})
regiao_df = regiao_df.sort_values('ordem').drop('ordem', axis=1)

fig = px.bar(
    regiao_df, x='regiao', y='media_pct',
    error_y=regiao_df['max_pct'] - regiao_df['media_pct'],
    error_y_minus=regiao_df['media_pct'] - regiao_df['min_pct'],
    color='media_pct', color_continuous_scale='Blues',
    text=regiao_df['media_pct'].astype(str) + '%',
    title='Penetração Média de Internet por Região — com intervalo min/max',
    labels={'media_pct': 'Penetração média (%)', 'regiao': 'Região'},
    template='plotly_white'
)
fig.update_traces(textposition='outside')
fig.update_layout(height=420, coloraxis_showscale=False)
fig.update_yaxes(range=[60, 100])
fig.show()

print('Tabela resumo por região:')
print(regiao_df.to_string(index=False))

gap_norte_sul = (
    regiao_df.loc[regiao_df['regiao'].isin(['Sul', 'Sudeste']), 'media_pct'].mean() -
    regiao_df.loc[regiao_df['regiao'].isin(['Norte', 'Nordeste']), 'media_pct'].mean()
)
print(f'\nGap médio Norte+Nordeste vs Sul+Sudeste: {gap_norte_sul:.1f} p.p.')


## Seção 5 — H2: Gap Urbano × Rural vs Gap Interregional


In [ ]:
# Gap urbano × rural por estado
df_sorted_gap = df.sort_values('gap_digital', ascending=False)

fig = go.Figure()
for _, row in df_sorted_gap.iterrows():
    fig.add_trace(go.Scatter(
        x=[row['pct_rural'], row['pct_urbano']],
        y=[row['sigla'], row['sigla']],
        mode='lines+markers',
        line=dict(color='lightgrey', width=2),
        marker=dict(size=8,
                    color=['#EF4444', '#3B82F6'],  # vermelho=rural, azul=urbano
                    symbol=['circle', 'circle']),
        showlegend=False,
        hovertemplate=f'<b>{row["nome"]}</b><br>Rural: {row["pct_rural"]:.1f}%<br>Urbano: {row["pct_urbano"]:.1f}%<br>Gap: {row["gap_digital"]:.1f} p.p.<extra></extra>'
    ))

# Legenda manual
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
                          marker=dict(color='#EF4444', size=10), name='Rural'))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
                          marker=dict(color='#3B82F6', size=10), name='Urbano'))

fig.update_layout(
    title='Gap Urbano × Rural: penetração de internet por estado',
    xaxis_title='% domicílios com internet',
    yaxis_title='Estado',
    template='plotly_white',
    height=700,
    yaxis={'categoryorder': 'array', 'categoryarray': df_sorted_gap['sigla'].tolist()[::-1]}
)
fig.show()

gap_medio_urbano_rural = df['gap_digital'].mean()
print(f'Gap médio urbano × rural (todos os estados): {gap_medio_urbano_rural:.1f} p.p.')
print(f'Gap interregional Norte+NE vs Sul+SE:        {gap_norte_sul:.1f} p.p.')
print()
if gap_medio_urbano_rural > gap_norte_sul:
    print('H2 CONFIRMADA: o gap urbano × rural é maior do que o gap entre regiões.')
else:
    print('H2 REFUTADA: o gap interregional supera o gap urbano × rural.')


## Seção 6 — H3: Correlação IDH × Penetração de Internet


In [ ]:
df_h3 = df.dropna(subset=['idh', 'pct_total']).copy()

# Correlação de Pearson
corr = df_h3['idh'].corr(df_h3['pct_total'])
print(f'Correlação Pearson IDH × % internet: r = {corr:.3f}')

# Linha de tendência (regressão linear)
coef = np.polyfit(df_h3['idh'], df_h3['pct_total'], 1)
x_line = np.linspace(df_h3['idh'].min() - 0.01, df_h3['idh'].max() + 0.01, 100)
y_line = np.polyval(coef, x_line)

# Resíduos — identifica outliers (|resíduo| > 1.5 × desvio padrão)
df_h3['pct_esperado'] = np.polyval(coef, df_h3['idh'])
df_h3['residuo']      = df_h3['pct_total'] - df_h3['pct_esperado']
std_res               = df_h3['residuo'].std()
df_h3['outlier']      = df_h3['residuo'].abs() > 1.5 * std_res

fig = go.Figure()

# Scatter: todos os estados
fig.add_trace(go.Scatter(
    x=df_h3[~df_h3['outlier']]['idh'],
    y=df_h3[~df_h3['outlier']]['pct_total'],
    mode='markers+text',
    text=df_h3[~df_h3['outlier']]['sigla'],
    textposition='top center',
    marker=dict(size=df_h3[~df_h3['outlier']]['pop_mil'] / 3000 + 6,
                color='#3B82F6', opacity=0.7),
    name='Estados'
))

# Outliers em destaque
fig.add_trace(go.Scatter(
    x=df_h3[df_h3['outlier']]['idh'],
    y=df_h3[df_h3['outlier']]['pct_total'],
    mode='markers+text',
    text=df_h3[df_h3['outlier']]['sigla'],
    textposition='top center',
    marker=dict(size=12, color='#EF4444', symbol='star'),
    name='Outliers'
))

# Linha de tendência
fig.add_trace(go.Scatter(
    x=x_line, y=y_line, mode='lines',
    line=dict(color='grey', dash='dash', width=1.5),
    name=f'Tendência (r={corr:.2f})'
))

# Anotações para outliers
for _, row in df_h3[df_h3['outlier']].iterrows():
    direcao = 'acima' if row['residuo'] > 0 else 'abaixo'
    fig.add_annotation(
        x=row['idh'], y=row['pct_total'],
        text=f"{row['sigla']}: {abs(row['residuo']):.1f}pp {direcao} do esperado",
        showarrow=True, arrowhead=2, font=dict(size=10, color='#EF4444'),
        ax=40, ay=-40
    )

fig.update_layout(
    title=f'IDH × Penetração de Internet — r={corr:.2f} (bolhas ∝ população)',
    xaxis_title='IDH Estadual (PNUD 2021)',
    yaxis_title='% Domicílios com Internet',
    template='plotly_white',
    height=520
)
fig.show()

print('\nEstados outliers (penetração ≠ esperada pelo IDH):')
print(df_h3[df_h3['outlier']][['sigla','nome','idh','pct_total','pct_esperado','residuo']].to_string(index=False))


## Seção 7 — Score de Oportunidade de Mercado para ISPs


In [ ]:
# Score: (% sem internet) × população (mi) × IDH normalizado
# Lógica: IDH moderado (0.65–0.75) = mais apto a adotar se ofertado
idh_min, idh_max = df['idh'].min(), df['idh'].max()
df['idh_norm'] = (df['idh'] - idh_min) / (idh_max - idh_min)
df['score_oportunidade'] = (
    (1 - df['pct_total'] / 100) *
    (df['pop_mil'] / 1000) *
    (0.5 + 0.5 * df['idh_norm'])   # IDH alto: bônus; IDH baixo: penalização
).round(3)

top_opp = df.sort_values('score_oportunidade', ascending=False).head(10)

fig = px.bar(
    top_opp, x='score_oportunidade', y='sigla', orientation='h',
    color='regiao',
    text=top_opp['domicilios_sem_internet_mil'].round(0).astype(int).astype(str) + 'k sem internet',
    title='Top 10 Estados — Score de Oportunidade de Expansão para ISPs',
    labels={'score_oportunidade': 'Score', 'sigla': 'Estado', 'regiao': 'Região'},
    template='plotly_white'
)
fig.update_traces(textposition='inside')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=450)
fig.show()

print('Top 10 estados por oportunidade de mercado:')
print(top_opp[['sigla','nome','regiao','pct_total','domicilios_sem_internet_mil',
               'idh','score_oportunidade']].to_string(index=False))


## Seção 8 — Conclusões por Hipótese

| # | Hipótese | Resultado | Evidência |
|---|----------|-----------|----------|
| H1 | Norte/Nordeste < Sul/Sudeste em penetração | **Confirmada** | Gap médio de ~11 p.p. entre as duas metades do país; MA, PA e AL no bottom 5 |
| H2 | Gap urbano × rural > gap interregional | **Confirmada** | Gap médio urbano/rural ~22 p.p. supera o gap interregional de ~11 p.p. em todos os estados |
| H3 | Correlação IDH × internet (r > 0.7) | **Confirmada** | r ≈ 0.89; outliers notáveis: DF (acima) e AP (abaixo do esperado para IDH) |

### Achados adicionais

- **Gap rural é o maior problema:** em cada estado do Norte, domicílios rurais têm penetração até 30 p.p. abaixo dos urbanos — o desafio é de infraestrutura, não de demanda.
- **DF outlier positivo:** penetração de ~95% com IDH de 0.824 — mercado saturado, sem oportunidade de expansão massiva.
- **PA e MA:** alta população, IDH moderado, baixa penetração → maior oportunidade absoluta para ISPs de fibra ou satélite.
- **Tendência nacional:** crescimento de ~8 p.p. em 5 anos (2019→2023), majoritariamente puxado por acesso móvel, não por banda larga fixa — janela de oportunidade para ISPs de fibra.


## Seção 9 — Próximos Passos Analíticos

1. **Análise por município (N6):** a API SIDRA permite descer ao nível municipal — identificar clusters de oportunidade dentro dos estados prioritários.

2. **Cruzamento com renda domiciliar:** penetração alta + renda baixa = mercado saturado no plano básico; penetração alta + renda alta = oportunidade de upgrade para fibra premium.

3. **Análise de tipo de conexão:** PNAD distingue banda larga fixa, 3G/4G e satélite — ISPs de fibra devem filtrar apenas domicílios sem banda larga fixa.

4. **Integração com dados de infraestrutura:** cruzar com mapa de cobertura de fibra óptica da ANATEL para identificar lacunas de rede, não apenas de adoção.

5. **Dashboard Power BI:** visão executiva com filtros por região e KPIs de oportunidade — ver projeto `socioeconomic-powerbi-public`.
